# AML 전처리 파이프라인

IBM AML 데이터셋(Kaggle) 전처리 노트북 — 텍스트 스크립트를 문단 단위로 셀 분할하여 변환.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq fonts-nanum


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import platform
import re
import shutil
import sys
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.csgraph import connected_components

SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', 120)
plt.rcParams['font.family'] = 'NanumGothic'  # 한글 라벨 글리프 누락 방지
plt.rcParams['axes.unicode_minus'] = False

# ── 사용자가 바꿀 핵심 설정 ───────────────────────────────────────────────────
ZIP_PATH = Path('/workspace/IBM_AML_dataset/archive (2).zip')
DATASET = 'HI-Small'          # HI/LI, Small/Medium/Large 를 합치지 않음(화두 7)
PROJECT_DIR = Path('/workspace')
RAW_DIR = Path('/workspace/IBM_AML_dataset')   # 이미 추출된 원본 CSV/TXT 위치
OUT_DIR = PROJECT_DIR / 'processed_multiclass'

# 수정 6 — 규모 모드. 'auto' 는 파일 크기로 고른다.
SCALE_MODE = 'auto'           # 'auto' | 'in_memory' | 'chunked'
CHUNKED_THRESHOLD_MIB = 2048  # 이 크기를 넘으면 chunked
CSV_CHUNK_ROWS = 2_000_000    # chunked 적재 단위
FEAT_BLOCK_ROWS = 1_000_000   # 피처 계산 블록 단위(memmap 출력)

# 수정 5 — 1차 이진 탐지용 전행 피처 저장(기본 True). 1차/2차 결과 비교에 필요.
SAVE_STAGE1_FULL = True

# 수정 3 — 고립 계좌: 표시와 제거를 분리한다.
MARK_ISOLATED = True          # 세탁 거래에서 그래프로 도달 불가능한 계좌를 '표시'
DROP_ISOLATED = False         # 실제 '제거'. True 로 켜면 평가 분포가 바뀐다(아래 경고 참조)

# 수정 4 — 달러 환산 피처
USD_FEATURES = True

# 수정 1 — 사건 구성 규칙
EVENT_RULES = ('attempt', 'window')
EVENT_WINDOW_MIN = 1440       # 시간창(분). **팀 미확정 가정값** — §10 참조

# 화두 10·17 재확인용 스위치: 절대 시각(hour/dow) 피처 사용 여부.
# 팀 결정 문구는 '절대 날짜·시각 대신 순서와 시간차'이므로, 끄면 문구에 정확히 맞는다.
USE_ABSOLUTE_TIME_FEATS = True

ALLOW_TRUNCATED_CSV = False   # 잘린 원본을 조용히 처리하지 않는다
# ─────────────────────────────────────────────────────────────────────────────

for _p in (RAW_DIR, OUT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

if DROP_ISOLATED:
    print('[warn] DROP_ISOLATED=True — 고립 계좌 마스크는 세탁 라벨에서 파생된다.\n'
          '       train 뿐 아니라 test 에서도 행이 사라지므로 평가 분포가 바뀐다.\n'
          '       "제거 전/후 비교 실험" 용도로만 켜고, 최종 성능 보고에는 두 경우를 모두 적는다.')

print({'python': sys.version.split()[0], 'platform': platform.platform(),
       'dataset': DATASET, 'out_dir': str(OUT_DIR), 'scale_mode': SCALE_MODE})


In [ ]:
def extract_member(zip_path: Path, member: str, raw_dir: Path) -> Path:
    target = raw_dir / member
    if target.exists() and target.stat().st_size > 0:
        print(f'[reuse] {target} ({target.stat().st_size / 2**20:,.1f} MiB)')
        return target
    if not zip_path.exists():
        raise FileNotFoundError(f'{zip_path} 없음 — 원본 CSV/TXT 를 {raw_dir} 에 직접 두거나 ZIP 을 다시 받는다')
    try:
        zf = zipfile.ZipFile(zip_path)
    except zipfile.BadZipFile as e:
        raise RuntimeError(
            f'{zip_path} 를 ZIP 으로 열 수 없다({e}). 다운로드가 중단된 파일일 가능성이 높다.\n'
            f'  - 원본 archive.zip 은 압축 상태로 약 8GiB 다.\n'
            f'  - 현재 크기: {zip_path.stat().st_size / 2**30:,.2f} GiB') from e
    with zf:
        info = zf.getinfo(member)
        print(f'[extract] {member}: {info.file_size / 2**20:,.1f} MiB')
        with zf.open(info) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst, length=8 * 2**20)
    return target


In [ ]:
def check_csv_complete(path: Path, n_fields: int = 11) -> dict:
    """파일 끝이 완결된 CSV 행인지 확인한다. 중단된 추출을 조용히 통과시키지 않는다."""
    size = path.stat().st_size
    with path.open('rb') as fh:
        fh.seek(max(0, size - 8192))
        last = fh.read().split(b'\n')[-1]
    complete = (last == b'') or (last.count(b',') == n_fields - 1)
    info = {'size_mib': round(size / 2**20, 1), 'complete': bool(complete),
            'tail': last[-48:].decode('utf-8', 'replace')}
    if not complete:
        msg = (f'[FAIL] {path.name} 의 마지막 줄이 잘려 있다 ({info["size_mib"]:,.1f} MiB).\n'
               f'       tail={info["tail"]!r}\n'
               f'       추출이 중간에 끊긴 파일이다. 다시 추출하기 전에는 이 세트로 학습하지 않는다.')
        if not ALLOW_TRUNCATED_CSV:
            raise RuntimeError(msg)
        print(msg + '\n[warn] ALLOW_TRUNCATED_CSV=True 라서 계속 진행한다.')
    else:
        print(f'[ok] {path.name} 무결성 통과 ({info["size_mib"]:,.1f} MiB)')
    return info


In [ ]:
TRANS_PATH = extract_member(ZIP_PATH, f'{DATASET}_Trans.csv', RAW_DIR)
PATTERNS_PATH = extract_member(ZIP_PATH, f'{DATASET}_Patterns.txt', RAW_DIR)
trans_check = check_csv_complete(TRANS_PATH)

if SCALE_MODE == 'auto':
    SCALE_MODE = 'chunked' if trans_check['size_mib'] > CHUNKED_THRESHOLD_MIB else 'in_memory'
print(f'[mode] SCALE_MODE={SCALE_MODE}')


In [ ]:
TAIL_MIN_FRAC = 0.05            # 일 거래량 < (최대 일 거래량 * 5%) 인 양끝 꼬리 일자 절단
TRAIN_FRAC, VAL_FRAC = 0.60, 0.20
W_1H, W_24H = 60, 1440          # 속도 피처 윈도 (분)
Z_CLIP, EPS = 10.0, 1e-6
NIGHT_END_HOUR = 6              # 야간 = 00:00 ~ 05:59
HIGH_RISK_FMT = ('Cash', 'Cheque')

RAW_DTYPE = {
    'From Bank': 'str', 'Account': 'str', 'To Bank': 'str', 'Account.1': 'str',
    'Amount Received': 'float64', 'Amount Paid': 'float64',
    'Receiving Currency': 'str', 'Payment Currency': 'str',
    'Payment Format': 'str', 'Is Laundering': 'int8',
}
RENAME = {
    'Timestamp': 'timestamp', 'From Bank': 'from_bank', 'Account': 'from_account',
    'To Bank': 'to_bank', 'Account.1': 'to_account',
    'Amount Received': 'amount_received', 'Receiving Currency': 'receiving_currency',
    'Amount Paid': 'amount_paid', 'Payment Currency': 'payment_currency',
    'Payment Format': 'payment_format', 'Is Laundering': 'is_laundering',
}
STR_COLS = ['from_bank', 'from_account', 'to_bank', 'to_account',
            'receiving_currency', 'payment_currency', 'payment_format']

CLASS_MAP = {'FAN-OUT': 0, 'FAN-IN': 1, 'CYCLE': 2, 'SCATTER-GATHER': 3,
             'GATHER-SCATTER': 4, 'BIPARTITE': 5, 'STACK': 6, 'RANDOM': 7}
OUT_OF_PATTERN, NORMAL = 8, -1
CLASS_NAMES = {0: 'FAN-OUT', 1: 'FAN-IN', 2: 'CYCLE', 3: 'SCATTER-GATHER',
               4: 'GATHER-SCATTER', 5: 'BIPARTITE', 6: 'STACK', 7: 'RANDOM',
               8: 'OUT_OF_PATTERN', -1: 'NORMAL'}
TRAIN_CLASSES = tuple(range(8))          # 수정 2 — 학습 타깃은 0~7 뿐

# 원핫 열 순서 고정용 표준 어휘. 실제 사전은 train 에 등장한 범주만 채택한다(train-only fit).
FMT_CANON = ['ACH', 'Bitcoin', 'Cash', 'Cheque', 'Credit Card', 'Reinvestment', 'Wire']
CCY_CANON = ['Australian Dollar', 'Bitcoin', 'Brazil Real', 'Canadian Dollar', 'Euro',
             'Mexican Peso', 'Ruble', 'Rupee', 'Saudi Riyal', 'Shekel', 'Swiss Franc',
             'UK Pound', 'US Dollar', 'Yen', 'Yuan']

# ── 수정 4: 고정 환율(1통화 단위당 USD). 팀 미확정 가정값 ─────────────────────
USD_RATE_AS_OF = '2024-06-01 (팀 미확정 가정값 — 확정 후 교체)'
USD_RATE = {
    'US Dollar': 1.0,        'Euro': 1.08,           'UK Pound': 1.27,
    'Swiss Franc': 1.12,     'Canadian Dollar': 0.73, 'Australian Dollar': 0.66,
    'Saudi Riyal': 0.267,    'Shekel': 0.267,        'Brazil Real': 0.19,
    'Mexican Peso': 0.055,   'Yuan': 0.138,          'Ruble': 0.0112,
    'Rupee': 0.0120,         'Yen': 0.0064,          'Bitcoin': 67000.0,
}
USD_RATE_FALLBACK = 1.0      # 미등재 통화는 1.0 으로 두고 meta 에 기록

# 패턴 역매칭 키. 문자열이 아니라 정수 ID 로 맞춘다(chunked 모드에서 문자열을 안 들고 있음).
JOIN_KEYS = ['ts_min', 'src_id', 'dst_id', 'cents', 'fmt_code']


In [ ]:
class KeyDict:
    """청크를 가로질러 '처음 나온 순서'로 정수 ID 를 부여한다. get_indexer 기반이라 빠르다."""

    def __init__(self, seed: list[str] | None = None):
        self.index = pd.Index(seed if seed else [], dtype=object)

    def encode(self, values) -> np.ndarray:
        vals = pd.Index(np.asarray(values, dtype=object))
        uniq = pd.unique(vals)
        missing = uniq[self.index.get_indexer(pd.Index(uniq)) < 0]
        if len(missing):
            self.index = self.index.append(pd.Index(missing, dtype=object))
        return self.index.get_indexer(vals).astype(np.int64, copy=False)

    def __len__(self) -> int:
        return len(self.index)


In [ ]:
def _prep_chunk(ch: pd.DataFrame, nodes: KeyDict, fmt: KeyDict,
                pay: KeyDict, recv: KeyDict) -> pd.DataFrame:
    """원본 청크 -> compact 컬럼. 문자열은 여기서 전부 정수로 바뀐다."""
    ch = ch.rename(columns=RENAME)
    for c in STR_COLS:
        ch[c] = ch[c].str.strip()
    ts = pd.to_datetime(ch['timestamp'], format='%Y/%m/%d %H:%M')
    src_key = (ch['from_bank'] + '_' + ch['from_account']).to_numpy(dtype=object)
    dst_key = (ch['to_bank'] + '_' + ch['to_account']).to_numpy(dtype=object)
    # 계좌 ID 는 '행 순서로 src, dst' 순서로 부여한다(두 모드가 같은 ID 순서를 쓰도록).
    inter = np.empty(2 * len(ch), dtype=object)
    inter[0::2], inter[1::2] = src_key, dst_key
    codes = nodes.encode(inter)
    return pd.DataFrame({
        'ts_epoch_min': ts.to_numpy(dtype='datetime64[m]').astype(np.int64),
        'src_id': codes[0::2].astype(np.int32),
        'dst_id': codes[1::2].astype(np.int32),
        'cents': np.rint(ch['amount_paid'].to_numpy() * 100).astype(np.int64),
        'recv_cents': np.rint(ch['amount_received'].to_numpy() * 100).astype(np.int64),
        'paid': ch['amount_paid'].to_numpy(dtype=np.float64),
        'recv': ch['amount_received'].to_numpy(dtype=np.float64),
        'fmt_code': fmt.encode(ch['payment_format']).astype(np.int16),
        'pay_code': pay.encode(ch['payment_currency']).astype(np.int16),
        'recv_code': recv.encode(ch['receiving_currency']).astype(np.int16),
        'is_pos': ch['is_laundering'].to_numpy(dtype=np.int8),
    })


In [ ]:
def load_compact(path: Path, mode: str) -> tuple[dict, dict]:
    """in_memory / chunked 공통 진입점. (raw 배열 묶음, 어휘·계좌 사전) 반환."""
    nodes = KeyDict()
    fmt, pay, recv = KeyDict(FMT_CANON), KeyDict(CCY_CANON), KeyDict(CCY_CANON)
    if mode == 'in_memory':
        ch = pd.read_csv(path, dtype=RAW_DTYPE)
        parts = [_prep_chunk(ch, nodes, fmt, pay, recv)]
        del ch
    else:
        part_dir = OUT_DIR / 'interim' / DATASET
        part_dir.mkdir(parents=True, exist_ok=True)
        for f in part_dir.glob('part-*.parquet'):
            f.unlink()                       # 계좌 사전과 짝이 맞아야 하므로 항상 재생성
        n_rows_seen = 0
        for i, ch in enumerate(pd.read_csv(path, dtype=RAW_DTYPE, chunksize=CSV_CHUNK_ROWS)):
            part = _prep_chunk(ch, nodes, fmt, pay, recv)
            part.to_parquet(part_dir / f'part-{i:05d}.parquet', index=False, compression='zstd')
            n_rows_seen += len(part)
            print(f'  [chunk {i:>3}] {len(part):,}행 누적 {n_rows_seen:,} / 계좌 {len(nodes):,}')
            del ch, part
            gc.collect()
        parts = [pd.read_parquet(f) for f in sorted(part_dir.glob('part-*.parquet'))]
    df = pd.concat(parts, ignore_index=True) if len(parts) > 1 else parts[0]
    del parts
    gc.collect()
    raw = {c: df[c].to_numpy() for c in df.columns}
    del df
    gc.collect()
    print(f'[load] {mode}: {len(raw["src_id"]):,}행 / 계좌 {len(nodes):,} / '
          f'수단 {len(fmt)} / 통화(지급) {len(pay)} / 통화(수취) {len(recv)}')
    return raw, {'nodes': nodes, 'fmt': fmt, 'pay': pay, 'recv': recv}


In [ ]:
def clean_rows(raw: dict) -> tuple[dict, dict]:
    """11개 원본 컬럼 완전 일치 중복만 제거(화두 11) + 비양수 금액 제거."""
    n0 = len(raw['src_id'])
    key_cols = ['ts_epoch_min', 'src_id', 'dst_id', 'cents', 'recv_cents',
                'fmt_code', 'pay_code', 'recv_code', 'is_pos']
    dup = pd.DataFrame({c: raw[c] for c in key_cols}).duplicated().to_numpy()
    bad = (raw['paid'] <= 0) | (raw['recv'] <= 0)
    keep = ~dup & ~bad
    n_dup, n_bad = int(dup.sum()), int((bad & ~dup).sum())
    n_dup_pos = int((raw['is_pos'][dup] == 1).sum())
    n_bad_pos = int((raw['is_pos'][bad & ~dup] == 1).sum())
    raw = {k: v[keep] for k, v in raw.items()}
    print(f'[clean] {n0:,}행 -> 완전중복 {n_dup:,}(양성 {n_dup_pos:,}) / '
          f'비양수금액 {n_bad:,}(양성 {n_bad_pos:,}) 제거 -> {len(raw["src_id"]):,}행')
    return raw, {'rows_in': n0, 'exact_duplicates': n_dup, 'exact_duplicates_positive': n_dup_pos,
                 'nonpositive_amount': n_bad, 'nonpositive_amount_positive': n_bad_pos,
                 'rows_out': int(len(raw['src_id']))}


In [ ]:
t0 = time.time()
raw, vocab_keys = load_compact(TRANS_PATH, SCALE_MODE)
raw, clean_info = clean_rows(raw)
# 모드 동등성 검증용 지문 — §14 에서 chunked/in_memory 결과를 바이트 단위로 비교한다.
RAW_FP = {k: hashlib.sha1(np.ascontiguousarray(v).tobytes()).hexdigest()[:16]
          for k, v in raw.items()}


In [ ]:
def trim_tails(raw: dict) -> tuple[dict, dict]:
    """일 거래량이 최대일의 TAIL_MIN_FRAC 미만인 양 끝단 꼬리 일자를 절단한다."""
    day = raw['ts_epoch_min'] // 1440
    uniq, cnt = np.unique(day, return_counts=True)
    thr = TAIL_MIN_FRAC * float(cnt.max())
    ok = uniq[cnt >= thr]
    lo, hi = int(ok.min()), int(ok.max())
    keep = (day >= lo) & (day <= hi)
    n_cut, n_cut_pos = int((~keep).sum()), int((raw['is_pos'][~keep] == 1).sum())
    d0 = np.datetime64(0, 'D') + np.timedelta64(lo, 'D')
    d1 = np.datetime64(0, 'D') + np.timedelta64(hi, 'D')
    print(f'[trim] 임계 {thr:,.0f}건/일 -> 유지 {d0}~{d1} ({hi - lo + 1}일), '
          f'절단 {n_cut:,}행(양성 {n_cut_pos:,})')
    return ({k: v[keep] for k, v in raw.items()},
            {'threshold_per_day': thr, 'kept_start': str(d0), 'kept_end': str(d1),
             'kept_days': hi - lo + 1, 'rows_cut': n_cut, 'positives_cut': n_cut_pos})


In [ ]:
def sort_split_reindex(raw: dict, n_nodes_raw: int) -> tuple[dict, np.ndarray, dict]:
    """시간순 정렬 -> 행수 60/20/20 분할(경계는 분 단위) -> 계좌 ID 압축 재부여."""
    order = np.argsort(raw['ts_epoch_min'], kind='stable')
    raw = {k: v[order] for k, v in raw.items()}
    ts = raw['ts_epoch_min']
    n = len(ts)
    t_val, t_te = ts[int(n * TRAIN_FRAC)], ts[int(n * (TRAIN_FRAC + VAL_FRAC))]
    assert t_val < t_te, '분할 경계 시각이 겹친다'
    split = np.full(n, 2, dtype=np.int8)
    split[ts < t_te] = 1
    split[ts < t_val] = 0
    raw['split'] = split
    cnt = np.bincount(split, minlength=3)

    used, inv = np.unique(np.concatenate([raw['src_id'], raw['dst_id']]), return_inverse=True)
    raw['src_id'], raw['dst_id'] = inv[:n].astype(np.int32), inv[n:].astype(np.int32)
    old2new = np.full(n_nodes_raw, -1, dtype=np.int32)
    old2new[used] = np.arange(len(used), dtype=np.int32)

    pair_key = raw['src_id'].astype(np.int64) * len(used) + raw['dst_id']
    _, pair_codes = np.unique(pair_key, return_inverse=True)
    raw['pair_id'] = pair_codes.astype(np.int64)
    raw['ts_min'] = (ts - ts.min()).astype(np.int64)

    dv = np.datetime64(0, 'm') + np.timedelta64(int(t_val), 'm')
    dt_ = np.datetime64(0, 'm') + np.timedelta64(int(t_te), 'm')
    print(f'[split] train {cnt[0]:,} / val {cnt[1]:,} / test {cnt[2]:,} (경계 {dv} / {dt_})')
    print(f'[ids] 계좌 {len(used):,} / 계좌쌍 {int(raw["pair_id"].max()) + 1:,}')
    return raw, old2new, {
        'val_start': str(dv), 'test_start': str(dt_), 'rows': cnt.tolist(),
        'fracs': [TRAIN_FRAC, VAL_FRAC, round(1 - TRAIN_FRAC - VAL_FRAC, 4)],
        'boundary_rule': '경계 분(minute)은 뒤 구간에 귀속',
        'n_accounts': int(len(used)), 'n_pairs': int(raw['pair_id'].max()) + 1,
        'ts_offset_epoch_min': int(ts.min())}


In [ ]:
n_nodes_raw = len(vocab_keys['nodes'])
raw, trim_info = trim_tails(raw)
raw, old2new, split_info = sort_split_reindex(raw, n_nodes_raw)
split, ts_all = raw['split'], raw['ts_min']
n_rows, n_nodes = len(ts_all), split_info['n_accounts']
gc.collect()


In [ ]:
# ── 필터 설정: 조합을 바꿔가며 비교한다 ──────────────────────────────────────
FILTERS = {
    # 세탁 거래에서 그래프로 무한 확장해도 닿지 않는 계좌의 거래 (2026-08-25 회의)
    'isolated_account': dict(enabled=True,  scope='train_only', mode='sample'),
    # 이웃이 수만 개인 준중앙은행 성격의 계좌 (화두 13, 교차검수 전이라 기본 off)
    'special_hub':      dict(enabled=False, scope='train_only', mode='sample'),
    # train 분위수 기준 극단 금액 (화두 9, 로그 변환 기준선과 비교용이라 기본 off)
    'extreme_amount':   dict(enabled=False, scope='train_only', mode='sample'),
}
SPECIAL_HUB_TOP_K = 15          # 화두 13 이 지목한 15개
EXTREME_AMOUNT_Q = 0.9999       # train 구간 분위수로만 적합한다
PROTECT_POSITIVES = True        # 양성(라벨1) 행은 어떤 필터로도 빼지 않는다
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
def flag_isolated(raw: dict, n_nodes: int) -> tuple[np.ndarray, dict]:
    """세탁 거래에서 도달 불가능한 계좌의 거래. 연결 컴포넌트 한 번으로 끝난다."""
    src, dst, pos = raw['src_id'], raw['dst_id'], raw['is_pos'] == 1
    g = coo_matrix((np.ones(len(src), dtype=np.int8), (src, dst)), shape=(n_nodes, n_nodes))
    n_comp, comp = connected_components(g, directed=False)
    hot = np.zeros(n_comp, dtype=bool)
    hot[comp[src[pos]]] = True                   # 세탁 엣지의 양 끝은 같은 컴포넌트
    node_bad = ~hot[comp]
    edge_bad = node_bad[src]                     # 엣지 양 끝은 항상 같은 컴포넌트
    touched = np.zeros(n_nodes, dtype=bool)
    touched[src] = True
    touched[dst] = True
    stats = {'n_components': int(n_comp),
             'accounts_total': int(touched.sum()),
             'accounts_flagged': int((touched & node_bad).sum()),
             'accounts_flagged_pct': round(100.0 * (touched & node_bad).sum()
                                           / max(int(touched.sum()), 1), 2)}
    assert not edge_bad[pos].any(), '세탁 거래가 고립으로 분류됐다 — 컴포넌트 계산 오류'
    return edge_bad, stats


In [ ]:
def flag_special_hub(raw: dict, n_nodes: int, k: int) -> tuple[np.ndarray, dict]:
    """차수 상위 k개 계좌에 닿는 거래."""
    deg = np.bincount(raw['src_id'], minlength=n_nodes) + \
          np.bincount(raw['dst_id'], minlength=n_nodes)
    hub = np.argsort(deg)[::-1][:k]
    is_hub = np.zeros(n_nodes, dtype=bool)
    is_hub[hub] = True
    edge_bad = is_hub[raw['src_id']] | is_hub[raw['dst_id']]
    return edge_bad, {'top_k': int(k), 'min_degree_in_topk': int(deg[hub].min()),
                      'max_degree': int(deg[hub].max()),
                      'accounts_flagged': int(k)}


In [ ]:
def flag_extreme_amount(raw: dict, split: np.ndarray, q: float) -> tuple[np.ndarray, dict]:
    """임계값은 train 구간에서만 적합한다(전체로 잡으면 그 자체가 누수)."""
    thr = float(np.quantile(raw['paid'][split == 0], q))
    return raw['paid'] > thr, {'quantile': q, 'threshold_paid': thr, 'fit_on': 'train only'}


In [ ]:
FLAG_FN = {'isolated_account': lambda: flag_isolated(raw, n_nodes),
           'special_hub': lambda: flag_special_hub(raw, n_nodes, SPECIAL_HUB_TOP_K),
           'extreme_amount': lambda: flag_extreme_amount(raw, split, EXTREME_AMOUNT_Q)}
SCOPE_MASK = {'train_only': lambda sp: sp == 0,
              'train_val': lambda sp: sp <= 1,
              'all': lambda sp: np.ones(len(sp), dtype=bool)}

filter_report, filter_meta = [], {}
graph_drop = np.zeros(n_rows, dtype=bool)
sample_drop = np.zeros(n_rows, dtype=bool)
is_pos_arr = raw['is_pos'] == 1

for name, cfg in FILTERS.items():
    if not cfg['enabled']:
        filter_meta[name] = {**cfg, 'applied_rows': 0}
        continue
    bad, st = FLAG_FN[name]()
    if PROTECT_POSITIVES:
        bad = bad & ~is_pos_arr
    applied = bad & SCOPE_MASK[cfg['scope']](split)
    (graph_drop if cfg['mode'] == 'graph' else sample_drop)[applied] = True
    per_split = [int((applied & (split == s)).sum()) for s in range(3)]
    filter_report.append({
        '필터': name, 'scope': cfg['scope'], 'mode': cfg['mode'],
        '해당 거래': int(bad.sum()), '해당 거래 %': round(100.0 * bad.sum() / n_rows, 2),
        '해당 계좌': st.get('accounts_flagged', np.nan),
        '해당 계좌 %': st.get('accounts_flagged_pct', np.nan),
        '적용 train': per_split[0], '적용 val': per_split[1], '적용 test': per_split[2],
        '양성 포함': int((applied & is_pos_arr).sum())})
    filter_meta[name] = {**cfg, 'flagged_rows': int(bad.sum()),
                         'flagged_pct': round(100.0 * bad.sum() / n_rows, 2),
                         'applied_rows': int(applied.sum()),
                         'applied_per_split': per_split, **st}

if filter_report:
    display(pd.DataFrame(filter_report))
else:
    print('[filter] 활성 필터 없음 — 원본 그대로 사용')
print('참고 — 2026-08-25 회의 보고치: HI-Small 고립 계좌 약 28% / 고립 거래 약 4%')

if graph_drop.any():
    keep = ~graph_drop
    raw = {k: v[keep] for k, v in raw.items()}
    sample_drop = sample_drop[keep]
    split, ts_all = raw['split'], raw['ts_min']
    n_rows = len(ts_all)
    is_pos_arr = raw['is_pos'] == 1
    print(f'[filter] mode=graph 로 {int(graph_drop.sum()):,}행 제거 -> {n_rows:,}행 '
          f'(이웃·이력 피처가 함께 바뀐다)')

sample_keep = ~sample_drop            # 학습 표본에 넣을 행
filter_meta['_summary'] = {
    'protect_positives': PROTECT_POSITIVES,
    'graph_dropped_rows': int(graph_drop.sum()),
    'sample_dropped_rows': int(sample_drop.sum()),
    'sample_kept_per_split': [int((sample_keep & (split == s)).sum()) for s in range(3)],
    'rows_per_split': [int((split == s).sum()) for s in range(3)]}
print({'표본 유지': filter_meta['_summary']['sample_kept_per_split'],
       '전체 행': filter_meta['_summary']['rows_per_split']})


In [ ]:
def load_patterns(path: Path, keys: dict, old2new: np.ndarray, ts_offset: int) -> pd.DataFrame:
    lines = pd.Series(Path(path).read_text(encoding='utf-8').splitlines())
    begin = lines.str.startswith('BEGIN LAUNDERING ATTEMPT')
    end_ = lines.str.startswith('END LAUNDERING ATTEMPT')
    hdr = lines.str.extract(r'^BEGIN LAUNDERING ATTEMPT - (.+)$', expand=False).ffill()
    ptype = hdr.str.extract(r'^([A-Z\-]+)', expand=False)
    data = (begin.cumsum() > end_.cumsum()) & ~begin & (lines.str.count(',') == 10)
    parts = lines.loc[data].str.split(',', expand=True)
    parts.columns = ['timestamp', 'from_bank', 'from_account', 'to_bank', 'to_account',
                     'amount_received', 'receiving_currency', 'amount_paid',
                     'payment_currency', 'payment_format', 'is_laundering']

    ts = pd.to_datetime(parts['timestamp'], format='%Y/%m/%d %H:%M')
    src_key = parts['from_bank'].str.strip() + '_' + parts['from_account'].str.strip()
    dst_key = parts['to_bank'].str.strip() + '_' + parts['to_account'].str.strip()
    node_idx = keys['nodes'].index

    def to_node(k: pd.Series) -> np.ndarray:
        old = node_idx.get_indexer(pd.Index(k.to_numpy(dtype=object)))
        return np.where(old >= 0, old2new[np.maximum(old, 0)], -1).astype(np.int32)

    fmt_idx = keys['fmt'].index
    pat = pd.DataFrame({
        'ts_min': ts.to_numpy(dtype='datetime64[m]').astype(np.int64) - ts_offset,
        'src_id': to_node(src_key),
        'dst_id': to_node(dst_key),
        'cents': np.rint(parts['amount_paid'].astype('float64').to_numpy() * 100).astype(np.int64),
        'fmt_code': fmt_idx.get_indexer(
            pd.Index(parts['payment_format'].str.strip().to_numpy(dtype=object))).astype(np.int16),
        'pattern': ptype.loc[data].str.strip().to_numpy(),
        # 헤더에 선언된 생성 파라미터(예: 'Max 13-degree Fan-Out'). 피처 검증용 정답값이다.
        'declared_param': pd.to_numeric(
            hdr.loc[data].str.extract(r'Max\s+(\d+)', expand=False), errors='coerce').to_numpy(),
        'attempt_id': (begin.cumsum().loc[data] - 1).to_numpy().astype(np.int32),
    })
    global PATTERN_ATTEMPTS
    PATTERN_ATTEMPTS = pat.groupby('attempt_id').agg(
        pattern=('pattern', 'first'), declared_param=('declared_param', 'first'),
        n_edges_declared=('pattern', 'size'))
    kinds = pd.Series(pat['pattern']).value_counts().to_dict()
    unknown = sorted(set(kinds) - set(CLASS_MAP))
    print(f'[pattern] {pat["attempt_id"].nunique():,}시도 / {len(pat):,}행 / 유형 {kinds}')
    if unknown:
        print(f'[pattern][warn] CLASS_MAP 에 없는 패턴명 {unknown} -> UNKNOWN_PATTERN 으로 분리 집계')
    return pat


In [ ]:
def attach_labels(a: dict, pat: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, dict]:
    """양성 행을 패턴 키로 역매칭해 y(0~8)를 만든다. 음성은 -1."""
    n = len(a['is_pos'])
    n_key_dup = int(pat.duplicated(JOIN_KEYS).sum())      # 동일 거래가 복수 시도에 속한 경우
    right = pat.drop_duplicates(JOIN_KEYS, keep='first')[JOIN_KEYS + ['pattern', 'attempt_id']]
    n_unmapped_acct = int(((right['src_id'] < 0) | (right['dst_id'] < 0)).sum())

    pos_idx = np.flatnonzero(a['is_pos'] == 1)
    left = pd.DataFrame({k: a[k][pos_idx] for k in JOIN_KEYS})
    merged = left.merge(right, on=JOIN_KEYS, how='left', validate='m:1')

    has_pat = merged['pattern'].notna().to_numpy()
    mapped = merged['pattern'].map(CLASS_MAP)
    cls = pd.to_numeric(mapped, errors='coerce').to_numpy(dtype='float64')
    n_unknown_name = int((has_pat & np.isnan(cls)).sum())
    cls = np.where(has_pat & np.isnan(cls), 7.0, cls)     # 이름은 출력했고 여기서만 7 로 둔다
    y_pos = np.where(has_pat, np.nan_to_num(cls), OUT_OF_PATTERN).astype(np.int8)

    y = np.full(n, NORMAL, dtype=np.int8)
    y[pos_idx] = y_pos
    attempt = np.full(n, -1, dtype=np.int32)
    attempt[pos_idx] = merged['attempt_id'].fillna(-1).astype(np.int32).to_numpy()

    n_matched = int(merged.loc[has_pat].drop_duplicates(JOIN_KEYS).shape[0])
    stats = {'pattern_rows': int(len(pat)), 'pattern_unique_keys': int(len(right)),
             'pattern_multi_attempt_keys': n_key_dup,
             'pattern_keys_with_unmapped_account': n_unmapped_acct,
             'matched_keys': n_matched, 'unmatched_keys': int(len(right)) - n_matched,
             'unknown_pattern_names': n_unknown_name,
             'positives': int(len(pos_idx)),
             'positives_in_pattern': int(has_pat.sum()),
             'positives_out_of_pattern': int((~has_pat).sum())}
    print(f'[label] 양성 {stats["positives"]:,}건 중 패턴 매칭 {stats["positives_in_pattern"]:,} / '
          f'OUT_OF_PATTERN {stats["positives_out_of_pattern"]:,} '
          f'(패턴 키 미매칭 {stats["unmatched_keys"]:,}, 계좌 미등재 {n_unmapped_acct:,})')
    return y, attempt, stats


In [ ]:
def usd_rate_vector(keys_index: pd.Index) -> tuple[np.ndarray, list[str]]:
    names = list(keys_index)
    miss = [nm for nm in names if nm not in USD_RATE]
    rates = np.array([USD_RATE.get(nm, USD_RATE_FALLBACK) for nm in names], dtype=np.float64)
    if miss:
        print(f'[usd][warn] 환율 미등재 통화 {miss} -> {USD_RATE_FALLBACK} 적용(meta 기록)')
    return rates, miss


In [ ]:
pat = load_patterns(PATTERNS_PATH, vocab_keys, old2new, split_info['ts_offset_epoch_min'])

a = {k: raw[k] for k in ('ts_min', 'src_id', 'dst_id', 'pair_id', 'paid', 'recv',
                         'cents', 'recv_cents', 'fmt_code', 'pay_code', 'recv_code', 'split')}
a['is_pos'] = (raw['is_pos'] == 1)
_day = raw['ts_epoch_min'] // 1440
a['hour'] = ((raw['ts_epoch_min'] % 1440) // 60).astype(np.int8)
a['dow'] = ((_day + 3) % 7).astype(np.int8)               # epoch day 0 = 1970-01-01 = 목요일
a['paid_log'] = np.log1p(a['paid'])
a['recv_log'] = np.log1p(a['recv'])
a['ccy_mismatch'] = a['pay_code'] != a['recv_code']
a['amt_mismatch'] = (~a['ccy_mismatch']) & (a['cents'] != a['recv_cents'])

pay_rate, pay_rate_miss = usd_rate_vector(vocab_keys['pay'].index)
recv_rate, recv_rate_miss = usd_rate_vector(vocab_keys['recv'].index)
a['usd_paid'] = a['paid'] * pay_rate[a['pay_code']]
a['usd_recv'] = a['recv'] * recv_rate[a['recv_code']]

del raw, _day
gc.collect()

y, attempt, label_stats = attach_labels(a, pat)
display(pd.Series(label_stats).to_frame('value'))


In [ ]:
class PastIndex:
    def __init__(self, ent: np.ndarray, ts: np.ndarray):
        ent = ent.astype(np.int64, copy=False)
        ts = ts.astype(np.int64, copy=False)
        self.bits = int(ts.max()).bit_length() + 1
        self.mask = (np.int64(1) << self.bits) - 1
        assert int(ent.max()) < (1 << (62 - self.bits)), 'ent<<B 가 int64 범위를 초과'
        order = np.lexsort((ts, ent))
        self.order = order.astype(np.int32) if len(order) < 2**31 else order
        self.key_s = (ent[order] << self.bits) | ts[order]
        del order

    def _key(self, ent: np.ndarray, ts: np.ndarray) -> np.ndarray:
        return (ent.astype(np.int64, copy=False) << self.bits) | ts.astype(np.int64, copy=False)

    def locate(self, ent: np.ndarray, ts: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        lo = np.searchsorted(self.key_s, ent.astype(np.int64, copy=False) << self.bits, side='left')
        hi = np.searchsorted(self.key_s, self._key(ent, ts), side='left')
        return lo, hi

    def window_lo(self, ent: np.ndarray, ts: np.ndarray, w: int) -> np.ndarray:
        t0 = np.maximum(ts.astype(np.int64, copy=False) - w, 0)
        return np.searchsorted(self.key_s, self._key(ent, t0), side='left')

    def prev_ts(self, lo: np.ndarray, hi: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """직전(엄격 과거) 거래 시각과 존재 플래그. 과거가 없으면 (-1, False)."""
        has = hi > lo
        prev = self.key_s[np.maximum(hi - 1, 0)] & self.mask
        return np.where(has, prev, -1), has

    def csum(self, x: np.ndarray) -> np.ndarray:
        """정렬 순서 prefix 합(선두 0). 구간 합 = csum[b] - csum[a]."""
        c = np.zeros(len(x) + 1, dtype=np.float64)
        np.cumsum(x[self.order].astype(np.float64, copy=False), out=c[1:])
        return c


In [ ]:
def fit_code_cols(codes: np.ndarray, tr: np.ndarray,
                  names_global: list[str]) -> tuple[np.ndarray, list[str]]:
    """train 에 등장한 범주만 열로 채택한다. 코드값이 오름차순 = 표준 어휘 순서."""
    seen = np.unique(codes[tr])
    seen = seen[seen >= 0]
    col_of = np.full(max(len(names_global), int(codes.max()) + 1), -1, dtype=np.int32)
    col_of[seen] = np.arange(len(seen), dtype=np.int32)
    return col_of, [names_global[c] for c in seen]


In [ ]:
def fit_group_stats(col: np.ndarray, x: np.ndarray, tr: np.ndarray,
                    k: int) -> tuple[np.ndarray, np.ndarray]:
    """그룹별 평균/표준편차(train 전용). 마지막 슬롯 = 미등재 폴백(전역 train 통계)."""
    m = tr & (col >= 0)
    cnt = np.bincount(col[m], minlength=k).astype(np.float64)
    s = np.bincount(col[m], weights=x[m], minlength=k)
    ss = np.bincount(col[m], weights=x[m] * x[m], minlength=k)
    g_mean, g_std = float(x[tr].mean()), float(x[tr].std())
    mean = np.where(cnt > 0, s / np.maximum(cnt, 1.0), g_mean)
    var = np.where(cnt > 1, ss / np.maximum(cnt, 1.0) - mean ** 2, g_std ** 2)
    return np.append(mean, g_mean), np.append(np.sqrt(np.maximum(var, 0.0)), g_std)


In [ ]:
def group_z(x_q: np.ndarray, col_q: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    idx = np.where(col_q >= 0, col_q, len(mean) - 1)          # 미등재 -> 폴백 슬롯
    return np.clip((x_q - mean[idx]) / (std[idx] + EPS), -Z_CLIP, Z_CLIP)


In [ ]:
def one_hot(col_q: np.ndarray, k: int) -> np.ndarray:
    oh = np.zeros((len(col_q), k), dtype=np.float32)
    r = np.flatnonzero(col_q >= 0)                            # 미등재 범주는 전부 0
    oh[r, col_q[r]] = 1.0
    return oh

class FeatureBuilder:
    """색인·통계를 한 번 만들고 transform(q) 로 블록별 피처를 낸다."""

    def __init__(self, a: dict, keys: dict):
        self.a = a
        tr = a['split'] == 0
        self.fmt_col, self.fmt_names = fit_code_cols(a['fmt_code'], tr, list(keys['fmt'].index))
        self.pay_col, self.pay_names = fit_code_cols(a['pay_code'], tr, list(keys['pay'].index))
        self.recv_col, self.recv_names = fit_code_cols(a['recv_code'], tr, list(keys['recv'].index))
        self.pay_mean, self.pay_std = fit_group_stats(
            self.pay_col[a['pay_code']], a['paid_log'], tr, len(self.pay_names))
        self.recv_mean, self.recv_std = fit_group_stats(
            self.recv_col[a['recv_code']], a['recv_log'], tr, len(self.recv_names))
        self.usd_mean = float(np.log1p(a['usd_paid'])[tr].mean())
        self.usd_std = float(np.log1p(a['usd_paid'])[tr].std())
        self.risk_cols = [self.fmt_col[c] for c, nm in enumerate(keys['fmt'].index)
                          if nm in HIGH_RISK_FMT and self.fmt_col[c] >= 0]

        print('[index] PastIndex 3종 구축 중…')
        self.i_src = PastIndex(a['src_id'], a['ts_min'])
        self.i_dst = PastIndex(a['dst_id'], a['ts_min'])
        self.i_pair = PastIndex(a['pair_id'], a['ts_min'])

        ps = self.i_pair.key_s >> self.i_pair.bits          # 정렬 순서의 pair_id
        first_sorted = np.empty(len(ps), dtype=bool)
        first_sorted[0] = True
        first_sorted[1:] = ps[1:] != ps[:-1]
        pair_first = np.empty(len(ps), dtype=np.float64)
        pair_first[self.i_pair.order] = first_sorted        # 그 쌍의 시간상 최초 등장 여부
        del ps, first_sorted

        print('[index] prefix 합 계산 중…')
        self.c_deg_s = self.i_src.csum(pair_first)
        self.c_deg_d = self.i_dst.csum(pair_first)
        del pair_first
        self.c1s, self.c2s = self.i_src.csum(a['paid_log']), self.i_src.csum(a['paid_log'] ** 2)
        self.c1d, self.c2d = self.i_dst.csum(a['recv_log']), self.i_dst.csum(a['recv_log'] ** 2)
        self.c_amt = self.i_src.csum(a['paid'])
        gc.collect()
        self.names, self.blocks = self._probe_names()
        print(f'[feat] {len(self.names)}차원 = ' +
              ' + '.join(f'{k} {v}' for k, v in self.blocks.items()))

    def _probe_names(self) -> tuple[list[str], dict]:
        _, names, blocks = self._compute(np.arange(min(8, len(self.a['ts_min']))), want_names=True)
        return names, blocks

    def _hist_z(self, c1, c2, lo, hi, x_q):
        cnt = (hi - lo).astype(np.float64)
        mean = (c1[hi] - c1[lo]) / np.maximum(cnt, 1.0)
        var = np.maximum((c2[hi] - c2[lo]) / np.maximum(cnt, 1.0) - mean ** 2, 0.0)
        z = np.where(cnt >= 2, (x_q - mean) / (np.sqrt(var) + EPS), 0.0)
        return np.clip(z, -Z_CLIP, Z_CLIP)

    def _compute(self, q: np.ndarray, want_names: bool = False):
        a = self.a
        feats: list[tuple[str, np.ndarray]] = []
        add = lambda nm, arr: feats.append((nm, np.asarray(arr, dtype=np.float32)))
        safe = lambda s: re.sub(r'\W+', '_', s)

        # ---- (A) 기본 엣지 피처 -------------------------------------------
        pay_q, recv_q = self.pay_col[a['pay_code'][q]], self.recv_col[a['recv_code'][q]]
        fmt_q = self.fmt_col[a['fmt_code'][q]]
        add('amt_paid_log', a['paid_log'][q])
        add('amt_recv_log', a['recv_log'][q])
        add('amt_z_pay_ccy', group_z(a['paid_log'][q], pay_q, self.pay_mean, self.pay_std))
        add('amt_z_recv_ccy', group_z(a['recv_log'][q], recv_q, self.recv_mean, self.recv_std))
        add('ccy_mismatch', a['ccy_mismatch'][q])
        add('amt_mismatch', a['amt_mismatch'][q])
        for prefix, col_q, nms in (('fmt', fmt_q, self.fmt_names),
                                   ('pccy', pay_q, self.pay_names),
                                   ('rccy', recv_q, self.recv_names)):
            oh = one_hot(col_q, len(nms))
            for j, v in enumerate(nms):
                add(f'{prefix}_{safe(v)}', oh[:, j])
        if USE_ABSOLUTE_TIME_FEATS:
            hr, dw = a['hour'][q].astype(np.float64), a['dow'][q].astype(np.float64)
            add('hour_sin', np.sin(2 * np.pi * hr / 24.0))
            add('hour_cos', np.cos(2 * np.pi * hr / 24.0))
            add('dow_sin', np.sin(2 * np.pi * dw / 7.0))
            add('dow_cos', np.cos(2 * np.pi * dw / 7.0))
            add('is_weekend', dw >= 5)
        n_base = len(feats)

        # ---- (A2) 달러 환산 피처 (수정 4) ----------------------------------
        if USD_FEATURES:
            up, ur = a['usd_paid'][q], a['usd_recv'][q]
            up_log = np.log1p(up)
            add('usd_paid_log', up_log)
            add('usd_recv_log', np.log1p(ur))
            add('usd_ratio_log', np.log((ur + 1.0) / (up + 1.0)))
            add('usd_gap_rel', np.clip(np.abs(up - ur) / (up + EPS), 0.0, 10.0))
            add('usd_paid_z', np.clip((up_log - self.usd_mean) / (self.usd_std + EPS),
                                      -Z_CLIP, Z_CLIP))
        n_usd = len(feats) - n_base

        # ---- PastIndex 조회 ------------------------------------------------
        sq, dq, pq, tq = a['src_id'][q], a['dst_id'][q], a['pair_id'][q], a['ts_min'][q]
        s_lo, s_hi = self.i_src.locate(sq, tq)
        d_lo, d_hi = self.i_dst.locate(dq, tq)
        p_lo, p_hi = self.i_pair.locate(pq, tq)
        s_w1, s_w24 = self.i_src.window_lo(sq, tq, W_1H), self.i_src.window_lo(sq, tq, W_24H)
        d_w1, d_w24 = self.i_dst.window_lo(dq, tq, W_1H), self.i_dst.window_lo(dq, tq, W_24H)
        p_w1, p_w24 = self.i_pair.window_lo(pq, tq, W_1H), self.i_pair.window_lo(pq, tq, W_24H)
        _, si_hi = self.i_dst.locate(sq, tq)                 # src 의 '수신' 이력(흐름 통과 신호)
        si_w24 = self.i_dst.window_lo(sq, tq, W_24H)
        _, do_hi = self.i_src.locate(dq, tq)                 # dst 의 '송신' 이력
        do_w24 = self.i_src.window_lo(dq, tq, W_24H)

        src_out_1h, src_out_24 = s_hi - s_w1, s_hi - s_w24
        dst_in_1h, dst_in_24 = d_hi - d_w1, d_hi - d_w24
        dt_src, s_has = self.i_src.prev_ts(s_lo, s_hi)
        dt_dst, d_has = self.i_dst.prev_ts(d_lo, d_hi)
        dt_pair, p_has = self.i_pair.prev_ts(p_lo, p_hi)
        dt_src = np.where(s_has, (tq - dt_src) * 60.0, 0.0)  # 초 단위
        dt_dst = np.where(d_has, (tq - dt_dst) * 60.0, 0.0)
        dt_pair = np.where(p_has, (tq - dt_pair) * 60.0, 0.0)

        # ---- (B) 상대 시간차·속도 피처 -------------------------------------
        add('dt_src_log', np.log1p(dt_src));   add('dt_src_first', ~s_has)
        add('dt_dst_log', np.log1p(dt_dst));   add('dt_dst_first', ~d_has)
        add('dt_pair_log', np.log1p(dt_pair)); add('dt_pair_first', ~p_has)
        add('src_out_cnt_1h', np.log1p(src_out_1h))
        add('src_out_cnt_24h', np.log1p(src_out_24))
        add('dst_in_cnt_1h', np.log1p(dst_in_1h))
        add('dst_in_cnt_24h', np.log1p(dst_in_24))
        add('pair_cnt_1h', np.log1p(p_hi - p_w1))
        add('pair_cnt_24h', np.log1p(p_hi - p_w24))
        add('src_in_cnt_24h', np.log1p(si_hi - si_w24))
        add('dst_out_cnt_24h', np.log1p(do_hi - do_w24))
        add('src_hist_out_deg', np.log1p(self.c_deg_s[s_hi] - self.c_deg_s[s_lo]))
        add('dst_hist_in_deg', np.log1p(self.c_deg_d[d_hi] - self.c_deg_d[d_lo]))
        n_dt = len(feats) - n_base - n_usd

        # ---- (C) 행동 이상치 피처 -------------------------------------------
        paid_q = a['paid'][q]
        add('tb_structuring_10k', np.exp(-((paid_q - 9800.0) ** 2) / (2.0 * 300.0 ** 2)))
        add('tb_structuring_50k', np.exp(-((paid_q - 49000.0) ** 2) / (2.0 * 1000.0 ** 2)))
        tb_z_src = self._hist_z(self.c1s, self.c2s, s_lo, s_hi, a['paid_log'][q])
        tb_z_dst = self._hist_z(self.c1d, self.c2d, d_lo, d_hi, a['recv_log'][q])
        add('tb_amt_z_src', tb_z_src)
        add('tb_amt_z_dst', tb_z_dst)
        add('tb_vel_src', np.log((src_out_1h + 1.0) * 24.0 / (src_out_24 + 24.0)))
        add('tb_vel_dst', np.log((dst_in_1h + 1.0) * 24.0 / (dst_in_24 + 24.0)))
        add('tb_dormant_burst',
            np.tanh((dt_src / 86400.0) / 3.0) * np.tanh(np.maximum(tb_z_src, 0.0) / 2.0))
        add('tb_night_cash_burst',
            (a['hour'][q] < NIGHT_END_HOUR) | np.isin(fmt_q, self.risk_cols))
        add('tb_self_loop', sq == dq)
        add('tb_round_amt', a['cents'][q] % 10000 == 0)
        out_sum_24 = self.c_amt[s_hi] - self.c_amt[s_w24]
        add('tb_amt_share_src_24h', paid_q / (paid_q + out_sum_24))
        add('tb_pair_repeat', np.log1p(p_hi - p_lo))
        n_tb = len(feats) - n_base - n_usd - n_dt

        names = [f for f, _ in feats]
        X = np.column_stack([v for _, v in feats]).astype(np.float32, copy=False)
        assert np.isfinite(X).all(), '피처에 NaN/inf 존재'
        assert int(src_out_24[~s_has].sum()) == 0, 'PastIndex 인과성 위반(첫 송금에 과거 건수)'
        blocks = {'edge_base': n_base, 'usd': n_usd, 'dt_velocity': n_dt, 'behavior': n_tb}
        return (X, names, blocks) if want_names else (X, None, None)

    def transform(self, q: np.ndarray) -> np.ndarray:
        return self._compute(q)[0]


In [ ]:
SPLIT_TAGS = ('tr', 'va', 'te')
STAGE1_DIR = OUT_DIR / 'stage1'
STAGE1_DIR.mkdir(parents=True, exist_ok=True)

fb = FeatureBuilder(a, vocab_keys)
FEATURE_NAMES, FEATURE_BLOCKS = fb.names, fb.blocks
n_feat = len(FEATURE_NAMES)

b1, b2 = int(np.searchsorted(split, 1)), int(np.searchsorted(split, 2))
SPLIT_BOUNDS = [(0, b1), (b1, b2), (b2, n_rows)]
scratch = OUT_DIR / '_scratch'
if not SAVE_STAGE1_FULL:
    scratch.mkdir(parents=True, exist_ok=True)

stage1_paths = {}
for s, (lo, hi) in enumerate(SPLIT_BOUNDS):
    tag = SPLIT_TAGS[s]
    dst_dir = STAGE1_DIR if SAVE_STAGE1_FULL else scratch
    path = dst_dir / f'X_{tag}.npy'
    mm = np.lib.format.open_memmap(path, mode='w+', dtype=np.float32, shape=(hi - lo, n_feat))
    for st in range(lo, hi, FEAT_BLOCK_ROWS):
        en = min(st + FEAT_BLOCK_ROWS, hi)
        mm[st - lo:en - lo] = fb.transform(np.arange(st, en))
    mm.flush()
    del mm
    stage1_paths[tag] = path
    if SAVE_STAGE1_FULL:
        np.save(STAGE1_DIR / f'y_{tag}.npy', a['is_pos'][lo:hi].astype(np.int8))
        # 수정 3 — 필터는 행을 지우지 않고 마스크로 남긴다. 조합을 바꿔도 X 를 다시 안 만든다.
        np.save(STAGE1_DIR / f'sample_mask_{tag}.npy', sample_keep[lo:hi])
    k = int(sample_keep[lo:hi].sum())
    print(f'[stage1] {tag}: X {hi - lo:,}x{n_feat} · 양성 {int(a["is_pos"][lo:hi].sum()):,} · '
          f'표본 유지 {k:,} ({100.0 * k / max(hi - lo, 1):.1f}%)')

def rows_of(global_idx: np.ndarray) -> np.ndarray:
    """전행 memmap 에서 임의의 전역 행 인덱스를 뽑는다(피처 재계산 없음)."""
    out = np.empty((len(global_idx), n_feat), dtype=np.float32)
    for s, (lo, hi) in enumerate(SPLIT_BOUNDS):
        m = (global_idx >= lo) & (global_idx < hi)
        if not m.any():
            continue
        mm = np.load(stage1_paths[SPLIT_TAGS[s]], mmap_mode='r')
        out[m] = mm[global_idx[m] - lo]
        del mm
    return out


In [ ]:
def build_events_attempt(y: np.ndarray, attempt: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """정답 시도 ID 기준. attempt<0(패턴 외 양성)은 이 규칙의 사건에 속하지 않는다."""
    idx = np.flatnonzero((y != NORMAL) & (attempt >= 0))
    _, comp = np.unique(attempt[idx], return_inverse=True)
    return idx, comp.astype(np.int32)


In [ ]:
def build_events_window(y: np.ndarray, a: dict, window: int) -> tuple[np.ndarray, np.ndarray]:
    """양성 거래를 '계좌 공유 + window 분 이내'로 이어붙인 연결 컴포넌트."""
    idx = np.flatnonzero(y != NORMAL)
    m = len(idx)
    ent = np.concatenate([a['src_id'][idx], a['dst_id'][idx]])
    eid = np.tile(np.arange(m, dtype=np.int32), 2)
    t = np.concatenate([a['ts_min'][idx], a['ts_min'][idx]])
    o = np.lexsort((t, ent))
    ent_s, eid_s, t_s = ent[o], eid[o], t[o]
    link = (ent_s[1:] == ent_s[:-1]) & ((t_s[1:] - t_s[:-1]) <= window)
    rows, cols = eid_s[:-1][link], eid_s[1:][link]
    g = coo_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(m, m))
    _, comp = connected_components(g, directed=False)
    return idx, comp.astype(np.int32)


In [ ]:
def label_events(comp: np.ndarray, y_mem: np.ndarray, n_comp: int) -> dict:
    """사건 라벨 = 구성 엣지 클래스의 최빈값. 전부 패턴 외면 8."""
    flat = np.bincount(comp.astype(np.int64) * 9 + y_mem, minlength=n_comp * 9)
    cnt = flat.reshape(n_comp, 9)
    known = cnt[:, :OUT_OF_PATTERN]
    tot = known.sum(axis=1)
    lab = np.where(tot > 0, known.argmax(axis=1), OUT_OF_PATTERN).astype(np.int8)
    purity = np.where(tot > 0, known.max(axis=1) / np.maximum(tot, 1), 1.0)
    return {'y': lab, 'purity': purity, 'n_oop_members': cnt[:, OUT_OF_PATTERN],
            'n_members': cnt.sum(axis=1)}


In [ ]:
def _longest_path(nn: int, ls: np.ndarray, ld: np.ndarray) -> int:
    """DAG 최장 경로 길이(홉). 사이클이 있으면 -1."""
    indeg = np.bincount(ld, minlength=nn).astype(np.int64)
    adj: list[list[int]] = [[] for _ in range(nn)]
    for u, v in zip(ls.tolist(), ld.tolist()):
        adj[u].append(v)
    stack = np.flatnonzero(indeg == 0).tolist()
    depth = [0] * nn
    seen = 0
    while stack:
        u = stack.pop()
        seen += 1
        for v in adj[u]:
            if depth[v] < depth[u] + 1:
                depth[v] = depth[u] + 1
            indeg[v] -= 1
            if indeg[v] == 0:
                stack.append(v)
    return max(depth) if seen == nn else -1


In [ ]:
def _is_bipartite(nn: int, ls: np.ndarray, ld: np.ndarray, has_self: bool) -> float:
    if has_self:
        return 0.0
    adj: list[list[int]] = [[] for _ in range(nn)]
    for u, v in zip(ls.tolist(), ld.tolist()):
        adj[u].append(v)
        adj[v].append(u)
    color = [-1] * nn
    for s in range(nn):
        if color[s] != -1:
            continue
        color[s] = 0
        stack = [s]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if color[v] == -1:
                    color[v] = 1 - color[u]
                    stack.append(v)
                elif color[v] == color[u]:
                    return 0.0
    return 1.0


In [ ]:
EVENT_STRUCT_NAMES = [
    'n_edges_log', 'n_nodes_log', 'edges_per_node', 'n_src_log', 'n_dst_log',
    'max_out_deg_log', 'max_in_deg_log', 'mean_out_deg', 'mean_in_deg',
    'fan_out_ratio', 'fan_in_ratio', 'pure_src_ratio', 'pure_dst_ratio', 'pass_ratio',
    'self_loop_ratio', 'unique_pair_ratio',
    'reciprocal_ratio', 'has_cycle', 'largest_scc_log', 'n_nontrivial_scc',
    'dag_depth', 'is_bipartite', 'src_dst_disjoint',
    'usd_total_log', 'usd_mean_log', 'usd_cv', 'usd_max_min_ratio_log',
    'flow_conservation', 'round_amt_ratio', 'amt_mismatch_ratio',
    'span_min_log', 'mean_dt_log', 'edges_per_hour_log', 'dt_cv', 'night_ratio',
    'n_unique_fmt', 'n_unique_pay_ccy', 'bitcoin_ratio', 'cash_cheque_ratio',
    'ccy_mismatch_ratio', 'reinvestment_ratio',
]
AGG_SPECS = [('amt_z_pay_ccy', 'mean'), ('amt_z_pay_ccy', 'max'),
             ('tb_vel_src', 'mean'), ('src_hist_out_deg', 'mean'),
             ('dst_hist_in_deg', 'mean'), ('tb_amt_share_src_24h', 'max')]
EVENT_FEATURE_NAMES = EVENT_STRUCT_NAMES + [f'agg_{how}_{nm}' for nm, how in AGG_SPECS]


In [ ]:
def event_features(mem_idx: np.ndarray, a: dict, code_ids: dict) -> np.ndarray:
    """사건 하나의 구조·금액·시간·범주 피처. 구성 엣지 안에서만 계산한다."""
    src, dst = a['src_id'][mem_idx], a['dst_id'][mem_idx]
    ts, up = a['ts_min'][mem_idx], a['usd_paid'][mem_idx]
    m = len(mem_idx)
    nodes, loc = np.unique(np.concatenate([src, dst]), return_inverse=True)
    nn = len(nodes)
    ls, ld = loc[:m].astype(np.int32), loc[m:].astype(np.int32)
    outd = np.bincount(ls, minlength=nn)
    ind = np.bincount(ld, minlength=nn)
    n_src, n_dst = int((outd > 0).sum()), int((ind > 0).sum())
    self_l = int((ls == ld).sum())
    pkey = ls.astype(np.int64) * nn + ld
    upair = np.unique(pkey)
    rkey = ld.astype(np.int64) * nn + ls
    recip = float(np.isin(upair, rkey).mean()) if len(upair) else 0.0

    g = csr_matrix((np.ones(m, dtype=np.int8), (ls, ld)), shape=(nn, nn))
    _, scc = connected_components(g, directed=True, connection='strong')
    sizes = np.bincount(scc)
    largest, n_nt = int(sizes.max()), int((sizes > 1).sum())
    cyc = 1.0 if (largest > 1 or self_l > 0) else 0.0
    depth = 0.0 if cyc else float(max(_longest_path(nn, ls, ld), 0))

    in_sum = np.bincount(ld, weights=up, minlength=nn)
    out_sum = np.bincount(ls, weights=up, minlength=nn)
    pas = (outd > 0) & (ind > 0)
    cons = float((np.minimum(in_sum[pas], out_sum[pas])
                  / (np.maximum(in_sum[pas], out_sum[pas]) + EPS)).mean()) if pas.any() else 0.0

    tsr = np.sort(ts)
    span = float(tsr[-1] - tsr[0])
    dts = np.diff(tsr).astype(np.float64) if m > 1 else np.zeros(1)
    fmt, payc = a['fmt_code'][mem_idx], a['pay_code'][mem_idx]

    return np.array([
        np.log1p(m), np.log1p(nn), m / nn, np.log1p(n_src), np.log1p(n_dst),
        np.log1p(outd.max()), np.log1p(ind.max()), m / max(n_src, 1), m / max(n_dst, 1),
        outd.max() / m, ind.max() / m,
        float(((outd > 0) & (ind == 0)).sum()) / nn, float(((ind > 0) & (outd == 0)).sum()) / nn,
        float(pas.sum()) / nn, self_l / m, len(upair) / m,
        recip, cyc, np.log1p(largest), float(n_nt), depth,
        _is_bipartite(nn, ls, ld, self_l > 0),
        1.0 if not np.intersect1d(src, dst, assume_unique=False).size else 0.0,
        np.log1p(up.sum()), np.log1p(up.mean()), float(up.std() / (up.mean() + EPS)),
        np.log1p(up.max() / (up.min() + EPS)), cons,
        float((a['cents'][mem_idx] % 10000 == 0).mean()),
        float(a['amt_mismatch'][mem_idx].mean()),
        np.log1p(span), np.log1p(dts.mean()), np.log1p(m / (span / 60.0 + 1.0)),
        float(dts.std() / (dts.mean() + EPS)),
        float((a['hour'][mem_idx] < NIGHT_END_HOUR).mean()),
        float(len(np.unique(fmt))), float(len(np.unique(payc))),
        float((payc == code_ids['ccy_bitcoin']).mean()),
        float(np.isin(fmt, code_ids['fmt_risk']).mean()),
        float(a['ccy_mismatch'][mem_idx].mean()),
        float((fmt == code_ids['fmt_reinvestment']).mean()),
    ], dtype=np.float64)


In [ ]:
# 사건 -> split 배정 정책. 'last_edge' = 완결 시점(기본), 'strict' = 경계 걸침 사건 제외.
# 'first_edge' 는 두지 않는다 — train 사건이 test 구간 엣지를 품게 되어 미래 정보 누수다.
EVENT_SPLIT_POLICY = 'last_edge'


In [ ]:
def build_event_dataset(rule: str) -> dict:
    """규칙 하나에 대해 사건 테이블 + 사건 피처 행렬을 만든다."""
    if rule == 'attempt':
        idx, comp = build_events_attempt(y, attempt)
    elif rule == 'window':
        idx, comp = build_events_window(y, a, EVENT_WINDOW_MIN)
    else:
        raise ValueError(rule)
    n_comp = int(comp.max()) + 1 if len(comp) else 0
    lab = label_events(comp, y[idx].astype(np.int64), n_comp)

    order = np.argsort(comp, kind='stable')
    starts = np.concatenate([[0], np.cumsum(np.bincount(comp, minlength=n_comp))])
    members = idx[order]

    agg_cols = [FEATURE_NAMES.index(nm) for nm, _ in AGG_SPECS]
    Xe = np.empty((n_comp, len(EVENT_FEATURE_NAMES)), dtype=np.float32)
    ev_split = np.empty(n_comp, dtype=np.int8)
    ev_bound = np.zeros(n_comp, dtype=np.int8)
    ev_t0 = np.empty(n_comp, dtype=np.int64)
    ev_t1 = np.empty(n_comp, dtype=np.int64)
    ev_keep = np.ones(n_comp, dtype=bool)

    edge_feats = rows_of(members)                       # 구성 엣지의 1차 피처(재계산 없음)
    for e in range(n_comp):
        sl = slice(starts[e], starts[e + 1])
        mem = members[sl]
        base = event_features(mem, a, CODE_IDS)
        blk = edge_feats[sl][:, agg_cols]
        agg = [blk[:, j].mean() if how == 'mean' else blk[:, j].max()
               for j, (_, how) in enumerate(AGG_SPECS)]
        Xe[e] = np.concatenate([base, np.asarray(agg, dtype=np.float64)])
        sp = split[mem]
        ev_split[e] = sp.max()                          # 사건 완결 시점 기준
        ev_bound[e] = int(sp.min() != sp.max())
        if EVENT_SPLIT_POLICY == 'strict' and ev_bound[e]:
            ev_keep[e] = False                          # 경계를 걸치는 사건은 아예 제외
        ev_t0[e], ev_t1[e] = a['ts_min'][mem].min(), a['ts_min'][mem].max()
        if not sample_keep[mem].all():                  # 필터에 걸린 엣지가 있으면 표본에서 제외
            ev_keep[e] = False

    assert np.isfinite(Xe).all(), '사건 피처에 NaN/inf 존재'
    meta = pd.DataFrame({
        'event_id': np.arange(n_comp, dtype=np.int32), 'y': lab['y'], 'split': ev_split,
        'n_members': lab['n_members'].astype(np.int32),
        'n_oop_members': lab['n_oop_members'].astype(np.int32),
        'purity': lab['purity'].astype(np.float32),
        'mixed': (lab['purity'] < 1.0).astype(np.int8),
        'boundary_incomplete': ev_bound, 'ts_start': ev_t0, 'ts_end': ev_t1,
        'sample_keep': ev_keep})
    print(f'[event:{rule}] 사건 {n_comp:,} / 구성 엣지 {len(members):,} / '
          f'평균 {len(members) / max(n_comp, 1):.2f}엣지 · 혼합 {int(meta["mixed"].sum()):,} · '
          f'경계걸침 {int(ev_bound.sum()):,} · 클래스8 {int((lab["y"] == OUT_OF_PATTERN).sum()):,}')
    return {'rule': rule, 'X': Xe, 'meta': meta, 'members': members,
            'offsets': starts.astype(np.int64)}


In [ ]:
CODE_IDS = {
    'ccy_bitcoin': int(vocab_keys['pay'].index.get_loc('Bitcoin'))
    if 'Bitcoin' in vocab_keys['pay'].index else -1,
    'fmt_reinvestment': int(vocab_keys['fmt'].index.get_loc('Reinvestment'))
    if 'Reinvestment' in vocab_keys['fmt'].index else -1,
    'fmt_risk': [int(vocab_keys['fmt'].index.get_loc(f)) for f in HIGH_RISK_FMT
                 if f in vocab_keys['fmt'].index],
}
events = {r: build_event_dataset(r) for r in EVENT_RULES}
display(pd.concat([e['meta'].assign(rule=r).groupby(['rule', 'y']).size()
                   .rename('사건 수').reset_index() for r, e in events.items()],
                  ignore_index=True).assign(
    클래스=lambda d: d['y'].map(CLASS_NAMES)))


In [ ]:
def save_split_arrays(dirpath: Path, X: np.ndarray, ylab: np.ndarray, spl: np.ndarray,
                      keep: np.ndarray, meta_df: pd.DataFrame | None = None) -> dict:
    """split 별로 X/y 를 저장한다. meta_df 를 주면 **X 행과 1:1로 정렬된** 메타도 같이 쓴다.

    events.parquet 은 event_id 순이고 X_{tag}.npy 는 split 순이라 그대로 join 하면 어긋난다.
    meta_{tag}.parquet 이 그 정렬 문제를 없앤다 — 같은 행 번호가 같은 사건이다.
    """
    dirpath.mkdir(parents=True, exist_ok=True)
    out = {}
    for s, tag in enumerate(SPLIT_TAGS):
        m = (spl == s) & keep
        np.save(dirpath / f'X_{tag}.npy', X[m])
        np.save(dirpath / f'y_{tag}.npy', ylab[m].astype(np.int64))
        if meta_df is not None:
            meta_df.loc[m].reset_index(drop=True).to_parquet(
                dirpath / f'meta_{tag}.parquet', index=False)
        out[tag] = {'shape': list(X[m].shape),
                    'class_hist': {CLASS_NAMES[int(c)]: int(v) for c, v in
                                   zip(*np.unique(ylab[m], return_counts=True))}}
    return out


In [ ]:
saved: dict = {'stage1': {t: {'shape': list(np.load(p, mmap_mode='r').shape)}
                          for t, p in stage1_paths.items()} if SAVE_STAGE1_FULL else {}}
OOP_DIR = OUT_DIR / 'out_of_pattern'

for rule, ev in events.items():
    md_ = ev['meta']
    trainable = (md_['y'].to_numpy() < OUT_OF_PATTERN) & md_['sample_keep'].to_numpy()
    oop = (md_['y'].to_numpy() == OUT_OF_PATTERN) & md_['sample_keep'].to_numpy()
    spl = md_['split'].to_numpy()
    saved[f'stage2_event_{rule}'] = save_split_arrays(
        OUT_DIR / f'stage2_event_{rule}', ev['X'], md_['y'].to_numpy(), spl, trainable,
        meta_df=md_)
    saved[f'out_of_pattern/event_{rule}'] = save_split_arrays(
        OOP_DIR / f'event_{rule}', ev['X'], md_['y'].to_numpy(), spl, oop, meta_df=md_)
    md_.to_parquet(OUT_DIR / f'stage2_event_{rule}' / 'events.parquet', index=False)
    np.savez_compressed(OUT_DIR / f'stage2_event_{rule}' / 'members.npz',
                        members=ev['members'], offsets=ev['offsets'])

# 엣지 단위 대조군 + 클래스 8 격리
pos_idx = np.flatnonzero(y != NORMAL)
X_pos = rows_of(pos_idx)
y_pos, spl_pos, keep_pos = y[pos_idx], split[pos_idx], sample_keep[pos_idx]
saved['stage2_edge'] = save_split_arrays(OUT_DIR / 'stage2_edge', X_pos, y_pos, spl_pos,
                                         keep_pos & (y_pos < OUT_OF_PATTERN))
saved['out_of_pattern/edge'] = save_split_arrays(OOP_DIR / 'edge', X_pos, y_pos, spl_pos,
                                                 keep_pos & (y_pos == OUT_OF_PATTERN))

np.savez_compressed(
    OUT_DIR / 'index_full.npz',
    y=y, split=split, is_laundering=a['is_pos'].astype(np.int8), attempt_id=attempt,
    ts_min=a['ts_min'].astype(np.int32), src_id=a['src_id'].astype(np.int32),
    dst_id=a['dst_id'].astype(np.int32), pair_id=a['pair_id'].astype(np.int64),
    sample_keep=sample_keep, pos_idx=pos_idx)
print('[save] 완료:', {k: {t: v[t]['shape'] for t in v} if k != 'stage1' else '전행'
                      for k, v in saved.items() if k != 'stage1'})

meta = {
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': DATASET,
    'source': {'trans': str(TRANS_PATH), 'patterns': str(PATTERNS_PATH),
               'trans_integrity': trans_check},
    'scale_mode': SCALE_MODE,
    'target': {
        'classes': {str(k): v for k, v in CLASS_NAMES.items()},
        'train_classes': list(TRAIN_CLASSES),
        'note': '학습 타깃은 0~7 뿐. 8(OUT_OF_PATTERN)은 out_of_pattern/ 으로 격리(수정 2). '
                '-1(NORMAL)은 1차 이진 타깃에서만 0 으로 쓴다.'},
    'rows_final': int(n_rows),
    'clean': clean_info,
    'trim': {**trim_info, 'tail_min_frac': TAIL_MIN_FRAC},
    'split': split_info,
    'filters': filter_meta,
    'labels': label_stats,
    'events': {r: {'rule': r,
                   'n_events': int(len(ev['meta'])),
                   'window_min': EVENT_WINDOW_MIN if r == 'window' else None,
                   'mean_members': float(ev['meta']['n_members'].mean()),
                   'mixed_events': int(ev['meta']['mixed'].sum()),
                   'boundary_incomplete': int(ev['meta']['boundary_incomplete'].sum()),
                   'class_hist': {CLASS_NAMES[int(c)]: int(v) for c, v in
                                  ev['meta']['y'].value_counts().items()}}
               for r, ev in events.items()},
    'edge_features': {'n': len(FEATURE_NAMES), 'names': FEATURE_NAMES,
                      'blocks': FEATURE_BLOCKS,
                      'absolute_time_feats': USE_ABSOLUTE_TIME_FEATS,
                      'windows_min': [W_1H, W_24H], 'z_clip': Z_CLIP},
    'event_features': {'n': len(EVENT_FEATURE_NAMES), 'names': EVENT_FEATURE_NAMES},
    'train_stats': {
        'currency_paid': {nm: {'mean': float(fb.pay_mean[i]), 'std': float(fb.pay_std[i])}
                          for i, nm in enumerate(fb.pay_names)},
        'currency_recv': {nm: {'mean': float(fb.recv_mean[i]), 'std': float(fb.recv_std[i])}
                          for i, nm in enumerate(fb.recv_names)},
        'fallback_paid': {'mean': float(fb.pay_mean[-1]), 'std': float(fb.pay_std[-1])},
        'usd_paid_log': {'mean': fb.usd_mean, 'std': fb.usd_std}},
    'usd': {'enabled': USD_FEATURES, 'as_of': USD_RATE_AS_OF, 'rates': USD_RATE,
            'fallback': USD_RATE_FALLBACK,
            'missing_pay': pay_rate_miss, 'missing_recv': recv_rate_miss},
    'files': saved,
    'leak_guards': [
        '시계열 60/20/20 분할, 경계 분 단위 정렬. 분할은 필터보다 먼저 확정한다',
        '통화 통계·범주 사전·극단값 임계 모두 train-only 적합(미등재는 폴백/0벡터)',
        'PastIndex: lexsort+searchsorted(side=left) 엄격 과거 참조, 동일 분 상호 불참조',
        '패턴 라벨·시도 ID·고립 마스크는 타깃 파생 메타데이터 — 피처 행렬에 미포함',
        '사건 split 은 마지막 엣지 시각 기준, 경계 걸침은 boundary_incomplete 로 표시'],
    'assumptions': [
        f'USD_RATE 고정 환율 {USD_RATE_AS_OF} — 팀 확정 전 가정값',
        f'EVENT_WINDOW_MIN={EVENT_WINDOW_MIN}분 — 후속 검증 9번(사건 생성 규칙) 미확정',
        f'꼬리 절단 임계 = 최대 일거래량의 {TAIL_MIN_FRAC:.0%} — 공식 주 기간 정의를 대신한 근사',
        'window 규칙의 연결은 이행적 — A-B, B-C 가 창 안이면 A-C 도 한 사건',
        '사건 라벨 = 구성 엣지 클래스의 최빈값, purity 로 섞임 정도 기록',
        f'PROTECT_POSITIVES={PROTECT_POSITIVES} — 양성 행은 필터로 제거하지 않는다'],
}
(OUT_DIR / 'features_meta.json').write_text(
    json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print(f'[save] features_meta.json -> {OUT_DIR.resolve()}')
print(f'[done] {time.time() - t0:,.1f}s')


In [ ]:
EXPECTED_HI_SMALL = {'label1_total': 5177, 'in_pattern': 3209, 'out_of_pattern': 1968,
                     'isolated_accounts_pct': 28.0, 'isolated_tx_pct': 4.0}
checks = []
if DATASET == 'HI-Small':
    exp = EXPECTED_HI_SMALL
    got_l1 = label_stats['positives'] + trim_info['positives_cut']
    got_pat = label_stats['positives_in_pattern'] + label_stats['unmatched_keys']
    iso = filter_meta.get('isolated_account', {})
    checks = [
        ('라벨1 거래 총계(꼬리 포함)', got_l1, exp['label1_total'], got_l1 == exp['label1_total']),
        ('8종 패턴 소속', got_pat, exp['in_pattern'], got_pat == exp['in_pattern']),
        ('패턴 외', label_stats['positives_out_of_pattern'], exp['out_of_pattern'],
         label_stats['positives_out_of_pattern'] == exp['out_of_pattern']),
        ('고립 계좌 %', iso.get('accounts_flagged_pct'), exp['isolated_accounts_pct'],
         abs((iso.get('accounts_flagged_pct') or 0) - exp['isolated_accounts_pct']) < 1.5),
        ('고립 거래 %', iso.get('flagged_pct'), exp['isolated_tx_pct'],
         abs((iso.get('flagged_pct') or 0) - exp['isolated_tx_pct']) < 1.0),
    ]
    display(pd.DataFrame(checks, columns=['항목', '이 노트북', '팀 문서', '일치'])
            .assign(일치=lambda d: d['일치'].map({True: 'O', False: 'X'})))
    if not all(c[3] for c in checks):
        print('[warn] 팀 문서와 어긋나는 항목이 있다. 파싱·필터 설정을 먼저 확인한다.')

print(f'[trim] 꼬리 절단으로 사라진 양성 {trim_info["positives_cut"]:,}건 = '
      f'패턴 키 미매칭 {label_stats["unmatched_keys"]:,}건 '
      f'(화두 17 후속검증 7번: 꼬리 제외 시 잘리는 세탁 체인·양성 라벨 수)')


In [ ]:
def verify_event_features() -> None:
    ev_a = events.get('attempt')
    if ev_a is None or not len(ev_a['meta']):
        print('[verify] attempt 규칙이 없어 건너뜀')
        return
    md_, memb, offs = ev_a['meta'], ev_a['members'], ev_a['offsets']
    ev_att = attempt[memb[offs[:-1]]]                     # 사건 -> 원본 시도 ID
    ref = PATTERN_ATTEMPTS.reindex(ev_att)
    declared = ref['declared_param'].to_numpy(dtype='float64')
    intact = ref['n_edges_declared'].to_numpy() == md_['n_members'].to_numpy()
    yv = md_['y'].to_numpy()
    F = lambda nm: ev_a['X'][:, EVENT_FEATURE_NAMES.index(nm)]

    rows = []
    def chk(label, mask, measured):
        m = mask & intact & ~np.isnan(declared)
        ok = np.isclose(measured[m], declared[m], atol=0.51)
        rows.append({'검증 항목': label, '대상 사건': int(m.sum()), '일치': int(ok.sum()),
                     '일치율': f'{100 * ok.mean():.0f}%' if m.sum() else '-'})
    chk('FAN-OUT 선언 차수 = max_out_deg', yv == 0, np.expm1(F('max_out_deg_log')))
    chk('FAN-IN 선언 차수 = max_in_deg', yv == 1, np.expm1(F('max_in_deg_log')))
    chk('CYCLE 선언 hops = 최대 SCC 크기', yv == 2, np.expm1(F('largest_scc_log')))
    chk('RANDOM 선언 hops = 구성 엣지 수', yv == 7,
        md_['n_members'].to_numpy().astype('float64'))
    display(pd.DataFrame(rows))
    if any(r['일치'] != r['대상 사건'] for r in rows):
        print('[FAIL] 정답 파라미터와 어긋나는 사건이 있다 — 사건 피처 계산을 먼저 고친다')

    # 엣지가 잘리면 구조 피처가 어떻게 무너지는가
    comp = []
    for cid, fn in ((2, 'has_cycle'), (5, 'is_bipartite'),
                    (0, 'fan_out_ratio'), (1, 'fan_in_ratio')):
        v, m = F(fn), yv == cid
        comp.append({'유형': CLASS_NAMES[cid], '피처': fn,
                     '엣지 온전': round(float(v[m & intact].mean()), 3) if (m & intact).any() else None,
                     '엣지 잘림': round(float(v[m & ~intact].mean()), 3) if (m & ~intact).any() else None,
                     '온전 n': int((m & intact).sum()), '잘림 n': int((m & ~intact).sum())})
    display(pd.DataFrame(comp))
    print('사건이 잘리면 CYCLE 의 순환 구조가 사라진다 — 경계에서 미완성인 사건은 '
          '같은 유형이라도 다른 분포다. 2차 모델은 boundary_incomplete 를 반드시 함께 봐야 한다.')


In [ ]:
verify_event_features()


In [ ]:
ev_summary = []
for rule, ev in events.items():
    m = ev['meta']
    for s, tag in enumerate(SPLIT_TAGS):
        sub = m[m['split'] == s]
        ev_summary.append({
            '규칙': rule, 'split': tag, '사건 수': len(sub),
            '학습 가능(0~7)': int(((sub['y'] < OUT_OF_PATTERN) & sub['sample_keep']).sum()),
            '클래스8': int((sub['y'] == OUT_OF_PATTERN).sum()),
            '경계 걸침': int(sub['boundary_incomplete'].sum()),
            '경계 걸침 %': round(100.0 * sub['boundary_incomplete'].mean(), 1) if len(sub) else 0.0,
            '평균 엣지': round(float(sub['n_members'].mean()), 2) if len(sub) else 0.0,
            '혼합 사건': int(sub['mixed'].sum())})
display(pd.DataFrame(ev_summary).set_index(['규칙', 'split']))

for rule, ev in events.items():
    m = ev['meta']
    tr_n = int(((m['split'] == 0) & (m['y'] < OUT_OF_PATTERN) & m['sample_keep']).sum())
    per_class = tr_n / len(TRAIN_CLASSES)
    flag = '  <-- 클래스당 20건 미만. 이 세트만으로 8종 분류를 학습·검증하기 어렵다' \
        if per_class < 20 else ''
    print(f'[warn:{rule}] train 학습 가능 사건 {tr_n:,}건 / 8클래스 = '
          f'클래스당 평균 {per_class:.1f}건{flag}')


In [ ]:
rows = []
for key, v in saved.items():
    if key == 'stage1':
        continue
    for tag, info in v.items():
        for cname, cnt in info['class_hist'].items():
            rows.append({'산출물': key, 'split': tag, '클래스': cname, '건수': cnt})
dist = pd.DataFrame(rows)
pivot = dist.pivot_table(index=['산출물', '클래스'], columns='split', values='건수',
                         aggfunc='sum', fill_value=0)[list(SPLIT_TAGS)]
display(pivot)


In [ ]:
ev_rules = list(events)
fig, axes = plt.subplots(len(ev_rules), 3, figsize=(15, 4 * len(ev_rules)), sharey='row')
axes = np.atleast_2d(axes)
BAR = '#4C72B0'
order = [CLASS_NAMES[c] for c in TRAIN_CLASSES] + [CLASS_NAMES[OUT_OF_PATTERN]]
for r, rule in enumerate(ev_rules):
    m = events[rule]['meta']
    for s, tag in enumerate(SPLIT_TAGS):
        ax = axes[r, s]
        sub = m.loc[m['split'] == s, 'y'].map(CLASS_NAMES).value_counts().reindex(order).fillna(0)
        bars = ax.bar(sub.index, sub.to_numpy(), color=BAR, width=0.7)
        ax.set_yscale('symlog')
        ax.set_title(f'{rule} · {tag}')
        ax.tick_params(axis='x', rotation=60)
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='y', color='#DDDDDD', linewidth=0.6)
        ax.set_axisbelow(True)
        for b, v in zip(bars, sub.to_numpy()):
            if v > 0:
                ax.text(b.get_x() + b.get_width() / 2, v, f'{int(v):,}',
                        ha='center', va='bottom', fontsize=8)
fig.suptitle('사건 단위 세탁 유형 분포 — 규칙 × 분할 (symlog y축, OUT_OF_PATTERN 은 학습 제외)')
fig.tight_layout()
plt.show()


In [ ]:
RUN_WINDOW_SWEEP = True
RUN_MODE_PARITY_CHECK = False

if RUN_WINDOW_SWEEP:
    sweep = []
    for w in (60, 360, 1440, 4320, 10080):
        i2, c2 = build_events_window(y, a, w)
        n2 = int(c2.max()) + 1
        l2 = label_events(c2, y[i2].astype(np.int64), n2)
        sweep.append({'window_min': w, '사건 수': n2,
                      '평균 엣지': round(float(l2['n_members'].mean()), 2),
                      '혼합 사건': int((l2['purity'] < 1.0).sum()),
                      '평균 순도': round(float(l2['purity'].mean()), 4),
                      '클래스8 사건': int((l2['y'] == OUT_OF_PATTERN).sum())})
    display(pd.DataFrame(sweep).set_index('window_min'))

if RUN_MODE_PARITY_CHECK:
    other = 'chunked' if SCALE_MODE == 'in_memory' else 'in_memory'
    raw2, _keys2 = load_compact(TRANS_PATH, other)
    raw2, _ci2 = clean_rows(raw2)
    fp2 = {k: hashlib.sha1(np.ascontiguousarray(v).tobytes()).hexdigest()[:16]
           for k, v in raw2.items()}
    diff = [k for k in RAW_FP if RAW_FP.get(k) != fp2.get(k)]
    print(f'[parity] {SCALE_MODE} vs {other}: '
          f'{"전 컬럼 일치" if not diff else "불일치 컬럼 " + str(diff)}')
    del raw2, fp2
    gc.collect()